# PPR501 - Homework 1

Bài làm gồm đủ 5 bài theo đề trong `PPR501_HW1.pdf`.

## Bài 1: Can Chi

Quy tắc theo đề:
- Can: lấy năm sinh chia 10, dùng phần dư để tra Can.
- Chi: lấy năm sinh chia 12, dùng phần dư để tra Chi.

In [1]:
def calculate_can_chi_calendar(year):
    can_by_remainder = ["Canh", "Tân", "Nhâm", "Quý", "Giáp", "Ất", "Bính", "Đinh", "Mậu", "Kỷ"]
    chi_by_remainder = ["Thân", "Dậu", "Tuất", "Hợi", "Tý", "Sửu", "Dần", "Mẹo", "Thìn", "Tỵ", "Ngọ", "Mùi"]

    can = can_by_remainder[year % 10]
    chi = chi_by_remainder[year % 12]
    return f"{can} {chi}"


test_years = [2025, 2024, 1997]
for year in test_years:
    print(f"calculate_can_chi_calendar({year}) -> {calculate_can_chi_calendar(year)}")


calculate_can_chi_calendar(2025) -> Ất Tỵ
calculate_can_chi_calendar(2024) -> Giáp Thìn
calculate_can_chi_calendar(1997) -> Đinh Sửu


## Bài 2: Tính giá trị tài khoản sau T năm

Cách tính:
- Mỗi năm có một lãi suất riêng.
- Lãi suất kép theo tháng: `monthly_rate = annual_rate / 12`.
- Mỗi tháng tính lãi trước, sau đó góp thêm `PMT` vào cuối tháng.

In [2]:
def calculate_account_value(principal, monthly_payment, years, annual_rates):
    if years > len(annual_rates):
        raise ValueError("Danh sách lãi suất phải có ít nhất T phần tử.")

    balance = principal
    for year_index in range(years):
        monthly_rate = annual_rates[year_index] / 12
        for _ in range(12):
            balance = balance * (1 + monthly_rate) + monthly_payment
    return balance


P = 100_000_000
PMT = 2_000_000
T = 5
annual_rates = [0.07, 0.075, 0.078, 0.08, 0.085]

final_value = calculate_account_value(P, PMT, T, annual_rates)
print(f"Tổng giá trị tài khoản sau {T} năm: {final_value:,.2f} VND")


Tổng giá trị tài khoản sau 5 năm: 294,277,728.99 VND


## Bài 3: Random hai số có tổng bằng 40

Chọn ngẫu nhiên hai số `x`, `y` trong khoảng `[1, 20]`, đếm số lần random cho đến khi `x + y = 40`.

In [3]:
import random


def random_number_with_condition(target_sum=40, low=1, high=20, seed=None):
    rng = random.Random(seed) if seed is not None else random
    count = 0

    while True:
        count += 1
        x = rng.randint(low, high)
        y = rng.randint(low, high)
        if x + y == target_sum:
            return count


result = random_number_with_condition(40, seed=0)
print(f"random_number_with_condition(40) -> {result} lần")


random_number_with_condition(40) -> 266 lần


## Bài 4: Class Stack

Xây dựng class `MyStack` với các method theo đề: `isEmpty`, `isFull`, `pop`, `push`, `top`.

In [4]:
class MyStack:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError("capacity phải là số nguyên dương.")
        self.capacity = capacity
        self.items = []

    def isEmpty(self):
        return len(self.items) == 0

    def isFull(self):
        return len(self.items) == self.capacity

    def pop(self):
        if self.isEmpty():
            raise IndexError("Stack đang rỗng, không thể pop.")
        return self.items.pop()

    def push(self, value):
        if self.isFull():
            raise OverflowError("Stack đã đầy, không thể push thêm.")
        self.items.append(value)

    def top(self):
        if self.isEmpty():
            raise IndexError("Stack đang rỗng, không có top element.")
        return self.items[-1]


stack1 = MyStack(capacity=5)
stack1.push(1)
stack1.push(2)

print(stack1.isFull())
print(stack1.top())
print(stack1.pop())
print(stack1.top())
print(stack1.pop())
print(stack1.isEmpty())


False
2
2
1
1
True


## Bài 5: RESTful API quản lý thư viện bằng Flask

API quản lý danh sách sách với định dạng `{id, title, author}` và có đủ chức năng:
- Thêm sách mới.
- Lấy danh sách sách.
- Cập nhật thông tin sách.
- Xoá sách.

In [5]:
from flask import Flask, jsonify, request


app = Flask(__name__, root_path=".")
books = []
next_book_id = 1


def find_book(book_id):
    return next((book for book in books if book["id"] == book_id), None)


@app.route("/books", methods=["GET"])
def get_books():
    return jsonify(books), 200


@app.route("/books", methods=["POST"])
def add_book():
    global next_book_id
    data = request.get_json() or {}
    title = data.get("title")
    author = data.get("author")

    if not title or not author:
        return jsonify({"error": "title và author là bắt buộc"}), 400

    book = {"id": next_book_id, "title": title, "author": author}
    books.append(book)
    next_book_id += 1
    return jsonify(book), 201


@app.route("/books/<int:book_id>", methods=["PUT"])
def update_book(book_id):
    book = find_book(book_id)
    if book is None:
        return jsonify({"error": "Không tìm thấy sách"}), 404

    data = request.get_json() or {}
    book["title"] = data.get("title", book["title"])
    book["author"] = data.get("author", book["author"])
    return jsonify(book), 200


@app.route("/books/<int:book_id>", methods=["DELETE"])
def delete_book(book_id):
    book = find_book(book_id)
    if book is None:
        return jsonify({"error": "Không tìm thấy sách"}), 404

    books.remove(book)
    return jsonify({"message": "Đã xoá sách", "book": book}), 200


with app.test_client() as client:
    response = client.post("/books", json={"title": "Python for Engineers", "author": "FSB"})
    print("POST /books:", response.status_code, response.get_json())

    response = client.post("/books", json={"title": "Clean Code", "author": "Robert C. Martin"})
    print("POST /books:", response.status_code, response.get_json())

    response = client.get("/books")
    print("GET /books:", response.status_code, response.get_json())

    response = client.put("/books/1", json={"title": "Python for Engineers - Updated"})
    print("PUT /books/1:", response.status_code, response.get_json())

    response = client.delete("/books/2")
    print("DELETE /books/2:", response.status_code, response.get_json())

    response = client.get("/books")
    print("GET /books:", response.status_code, response.get_json())


POST /books: 201 {'author': 'FSB', 'id': 1, 'title': 'Python for Engineers'}
POST /books: 201 {'author': 'Robert C. Martin', 'id': 2, 'title': 'Clean Code'}
GET /books: 200 [{'author': 'FSB', 'id': 1, 'title': 'Python for Engineers'}, {'author': 'Robert C. Martin', 'id': 2, 'title': 'Clean Code'}]
PUT /books/1: 200 {'author': 'FSB', 'id': 1, 'title': 'Python for Engineers - Updated'}
DELETE /books/2: 200 {'book': {'author': 'Robert C. Martin', 'id': 2, 'title': 'Clean Code'}, 'message': 'Đã xoá sách'}
GET /books: 200 [{'author': 'FSB', 'id': 1, 'title': 'Python for Engineers - Updated'}]
